# KOA GUI (contratos) com Router e Map-Reduce (perguntas globais)

## Objetivo
Este notebook implementa uma interface (Gradio) para perguntas e respostas sobre **contratos representados como fatos Prolog** (um arquivo por contrato), com:

- **Grounding formal**: respostas devem ser justificadas por evidências presentes nos fatos.
- **Personas**: modo de resposta controlado (Advogado, Gestor, Executivo, Técnico).
- **Router**: decide se a pergunta é:
  1) direta sobre um contrato específico (DIRECT),
  2) sobre requisitos semânticos (SEMREQ),
  3) global (MAP-REDUCE), quando exige olhar múltiplos contratos.

## Princípio importante
Existe **um único contrato epistemológico** com o modelo (regras + grounding + persona).
A estratégia **Map-Reduce não cria um novo prompt**: ela apenas reaplica o mesmo contrato por arquivo (MAP) e agrega os resultados (REDUCE).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip uninstall google-generativeai -y
!pip install -q google-genai

In [ ]:
# Importa bibliotecas padrão e do Gemini
import os
from pathlib import Path
import re
import json

import yaml
import time

from typing import Any, List, Optional, Tuple
from dataclasses import dataclass

from google import genai

import gradio as gr

In [ ]:
# Recupera chave Gemini com secrets do Colab
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Configura pastas e arquivos
directory = '/content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/'
persist_directory = '/content/drive/My Drive/KOA/onboarding/modelos/'

PL_FOLDER = directory
SEMANTIC_GROUND_PATH = str(Path(persist_directory) / "KOA_semantic_ground.pl")


## 2) Carregamento dos dados (contratos em Prolog + Semantic Ground)

- Cada contrato está em um arquivo `.pl`.
- O **Semantic Ground** contém definições/regras do domínio que orientam interpretações (ex.: critérios de risco, conceitos etc.).

In [ ]:
# Carrega os contratos (arquivos .pl) como anexos de texto.
@dataclass
class AttachmentText:
    filename: str
    text: str

def load_pl_attachments(folder: str, pattern: str = "KOA_UFO_contrato_ocs_*.pl") -> List[AttachmentText]:
    p = Path(folder)
    attachments: List[AttachmentText] = []
    for fp in sorted(p.glob(pattern), key=lambda x: x.name.lower()):
        content = fp.read_text(encoding="utf-8", errors="replace")
        attachments.append(AttachmentText(filename=fp.name, text=content))
    return attachments

ATTACHMENTS = load_pl_attachments(PL_FOLDER)

print("Loaded .pl files:", len(ATTACHMENTS))
if not ATTACHMENTS:
    raise RuntimeError(f"No .pl files found in: {PL_FOLDER}")


In [ ]:
# Carrega o Semantic Ground (fatos Prolog com definições/regras do domínio).
def read_pl_as_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

SEMANTIC_GROUND_PL = read_pl_as_text(SEMANTIC_GROUND_PATH)

## 3) Funções auxiliares (roteamento, parsing e comandos)

Este bloco contém utilitários para:
- detectar comandos especiais na caixa de texto,
- normalizar perguntas,
- identificar quando usar SEMREQ vs DIRECT vs MAP-REDUCE.

In [ ]:
# Funções de apoio
import re
from typing import List, Optional, Tuple, Dict

# OCS
_OCS_PATTERNS = [
    re.compile(r"\bOCS\s*0*(\d{1,4})\s*/\s*(\d{4})\b", re.IGNORECASE),
    re.compile(r"\bOCS[\s_\-]*0*(\d{1,4})[\s_\-]*(\d{4})\b", re.IGNORECASE),
]

def normalize_ocs(n: int, y: int) -> str:
    return f"OCS_{int(n):03d}_{int(y)}"

def extract_ocs_from_text(text: str) -> Optional[str]:
    for pat in _OCS_PATTERNS:
        m = pat.search(text or "")
        if m:
            return normalize_ocs(int(m.group(1)), int(m.group(2)))
    return None

def last_ocs_from_history(history) -> Optional[str]:
    """
    history: lista [(user, assistant), ...] do Gradio
    Procura o OCS mais recente nas últimas mensagens do usuário.
    """
    if not history:
        return None
    # varre do fim para o começo
    for user_msg, _ in reversed(history[-10:]):
        ocs = extract_ocs_from_text(user_msg or "")
        if ocs:
            return ocs
    return None

# Marcadores dêiticos
_DEICTIC_PATTERNS = [
    r"\b(esse|essa|esses|essas)\b",
    r"\b(desse|dessa|desses|dessas)\b",
    r"\b(nesse|nessa|nesses|nessas)\b",
    r"\b(este|esta|estes|estas)\b",
    r"\b(deste|desta|destes|destas)\b",
    r"\b(neste|nesta|nestes|nestas)\b",
    r"\b(aqui|acima|abaixo|anterior|seguinte)\b",
    r"\b(último|ultima|última|recent(e|es)|mais\s+recente)\b",
    # common in your domain
    r"\b(neste\s+contrato|nesse\s+contrato|desse\s+contrato|este\s+contrato|esse\s+contrato)\b",
]
_DEICTIC_RE = re.compile("|".join(_DEICTIC_PATTERNS), re.IGNORECASE)

def is_deictic_question_pt(text: str) -> bool:
    return bool(_DEICTIC_RE.search(text or ""))

def build_contract_index(attachments) -> Dict[str, Any]:
    """
    Índice: 'OCS_048_2022' -> AttachmentText
    Ajusta o regex do filename conforme seu padrão.
    """
    idx = {}
    for a in attachments:
        m = re.search(r"OCS_(\d{3})_(\d{4})", a.filename, flags=re.IGNORECASE)
        if m:
            key = f"OCS_{int(m.group(1)):03d}_{int(m.group(2))}"
            idx[key] = a
    return idx

CONTRACT_INDEX = build_contract_index(ATTACHMENTS)

# JSON helper: às vezes o modelo devolve texto extra; esta função tenta extrair um objeto JSON de forma robusta.
def _safe_json_loads(text: str) -> Optional[dict]:
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                return None
    return None


In [ ]:
# Funções para tratamento de comandos na interface
def has_semreq_indicator(text: str) -> bool:
    t = (text or "").strip().lower()
    return t.startswith("semreq:") or t.startswith("sr:") or t.startswith("/sr") or t.startswith("/semreq")

def has_all_indicator(text: str) -> bool:
    t = (text or "").strip().lower()
    # aceita /all no começo ou em qualquer lugar como token
    return bool(re.search(r"(^|\s)/all(\s|$)", t))

def select_attachments_for_question(question, attachments, history, last_selected_filename):
    # /sr → SEMREQ
    # /all → GLOBAL
    # senão → DIRECT (com fallback para “sem contrato identificado”)

    # SEMREQ explicit
    if has_semreq_indicator(question):
        return ([], last_selected_filename, "SEMREQ")

    # GLOBAL só com /all
    if has_all_indicator(question):
        return (attachments, last_selected_filename, "GLOBAL")

    # A partir daqui, NUNCA será global
    # 1) OCS explícito na pergunta
    q_ocs = extract_ocs_from_text(question)
    if q_ocs and q_ocs in CONTRACT_INDEX:
        a = CONTRACT_INDEX[q_ocs]
        return ([a], a.filename, "DIRECT")

    # 2) Deíctico -> tenta histórico / last_selected
    if is_deictic_question_pt(question):
        h_ocs = last_ocs_from_history(history)
        if h_ocs and h_ocs in CONTRACT_INDEX:
            a = CONTRACT_INDEX[h_ocs]
            return ([a], a.filename, "DIRECT")

        if last_selected_filename:
            a = next((x for x in attachments if x.filename == last_selected_filename), None)
            if a:
                return ([a], a.filename, "DIRECT")

        # Sem contexto suficiente -> DIRECT vazio (vai pedir OCS)
        return ([], last_selected_filename, "DIRECT")

    # 3) Sem deíctico: se só tiver um contrato carregado, usa ele
    if len(attachments) == 1:
        return ([attachments[0]], attachments[0].filename, "DIRECT")

    # 4) Mais de um contrato e nenhum OCS/deíctico -> não adivinha: pede OCS
    return ([], last_selected_filename, "DIRECT")


## 4) Prompt base e Personas (contrato epistemológico)

Este é o **prompt canônico** do KOA neste notebook:
- regras de grounding,
- regras de evidência,
- personas (estilo de resposta).

⚠️ Importante: os prompts do Map-Reduce reutilizam exatamente estas regras e a persona selecionada.

In [ ]:
PERSONA_RESPONSE_GUIDELINES = {
    "executivo": {
        "format": "texto_continuo",
        "detail_level": "baixo",
        "evidence_priority": "nenhuma",
        "show_clauses": False,
        "recommendations": False,
        "explain_reasoning": False
    },
    "advogado": {
        "format": "texto_analitico",
        "detail_level": "alto",
        "evidence_priority": "ufo_hohfeld",
        "show_clauses": True,
        "recommendations": False,
        "explain_reasoning": False
    },
    "gestor": {
        "format": "texto_explicativo",
        "detail_level": "medio",
        "evidence_priority": "clausulas",
        "show_clauses": True,
        "recommendations": True,
        "explain_reasoning": False
    },
    "tecnico": {
        "format": "texto_tecnico",
        "detail_level": "alto",
        "evidence_priority": "pipeline",
        "show_clauses": True,
        "recommendations": False,
        "explain_reasoning": True
    }
}

## Personas como *Single Source of Truth*

A partir desta seção, **todas** as regras de persona (prompt + apresentação da resposta) são definidas em **um único dicionário** (`PERSONA_SOT`).  
Os dicionários `PERSONA_GUIDANCE_EN` e `PERSONA_RESPONSE_GUIDELINES` passam a ser **derivados automaticamente**, apenas por compatibilidade.

**Objetivo:** evitar conflitos e garantir consistência entre o prompt e o pós-processamento.

In [ ]:
# ============================================================
# Personas: Single Source of Truth (SoT)
# ============================================================
# Canonical keys: executivo, advogado, gestor, tecnico
# Everything (prompt guidance + post-processing rules) is derived from PERSONA_SOT.

PERSONA_SOT = {
    "executivo": {
        "label_pt": "Executivo de negócio",
        "label_en": "Business Executive",
        "response": {
            "avoid_bullets": True,
            "prefer_prose": True,
            "detail_level": "baixo",
            "show_clauses": False,           # only if user explicitly asks
            "evidence_priority": "impacto",
            "recommendations": False,
            "explain_reasoning": False,
        },
        "prompt_guidance_en": (
            "Persona: Business Executive (Executivo de negócio).\n"
            "- Provide executive, decision-support answers in continuous prose: start with the conclusion, then risk/impact, then minimal context.\n"
            "- Do NOT use bullet points unless strictly necessary for disambiguation.\n"
            "- Do NOT mention Prolog, LLMs, prompts, routing, code, or any technical aspect of the solution.\n"
            "- Do NOT quote clauses or show clause identifiers unless the user explicitly asks for clause-level evidence.\n"
            "- If evidence is needed, paraphrase at a high level (no clause IDs) and keep it very short."
        ),
    },
    "advogado": {
        "label_pt": "Advogado",
        "label_en": "Lawyer",
        "response": {
            "avoid_bullets": True,
            "prefer_prose": True,
            "detail_level": "alto",
            "show_clauses": True,
            "evidence_priority": "ufo_hohfeld",
            "recommendations": False,
            "explain_reasoning": False,
        },
        "prompt_guidance_en": (
            "Persona: Lawyer (Advogado).\n"
            "- Provide answers, justifications, and detailed evidence from a LEGAL perspective grounded in the ontology.\n"
            "- PRIORITIZE (when available) Hohfeld/UFO normative positions (right, duty, power, liability, immunity, disability) as the primary justification.\n"
            "- Use contract clauses as textual support (secondary), referencing clause identifiers and paraphrasing/quoting only what is strictly relevant.\n"
            "- Do NOT use bullet points unless strictly necessary for clarity.\n"
            "- Do NOT mention Prolog, LLMs, prompts, routing, code, or any technical aspect of the solution."
        ),
    },
    "gestor": {
        "label_pt": "Gestor de Contrato",
        "label_en": "Contract Manager",
        "response": {
            "avoid_bullets": True,
            "prefer_prose": True,
            "detail_level": "medio",
            "show_clauses": True,
            "evidence_priority": "clausulas",
            "recommendations": True,
            "explain_reasoning": False,
        },
        "prompt_guidance_en": (
            "Persona: Contract Manager (Gestor de Contrato).\n"
            "- Provide answers, justifications, and evidence focusing on contract management and operational responsibilities.\n"
            "- PRIORITIZE contract clauses as primary evidence, referencing clause identifiers and paraphrasing/quoting only what is relevant.\n"
            "- When applicable, end with explicit, actionable recommendations for the Contract Manager.\n"
            "- Do NOT use bullet points unless strictly necessary for clarity.\n"
            "- Do NOT mention Prolog, LLMs, prompts, routing, code, or any technical aspect of the solution."
        ),
    },
    "tecnico": {
        "label_pt": "Técnico",
        "label_en": "Technical",
        "response": {
            "avoid_bullets": False,  # trace may need structured formatting
            "prefer_prose": True,
            "detail_level": "alto",
            "show_clauses": True,
            "evidence_priority": "pipeline",
            "recommendations": False,
            "explain_reasoning": True,
        },
        "prompt_guidance_en": (
            "Persona: Technical (Técnico).\n"
            "- Provide a concise answer PLUS as much technical trace as possible about how the answer was produced.\n"
            "- You MAY mention Prolog facts, predicates, rules, routing decisions (SEMREQ vs DIRECT vs MAP-REDUCE), and limitations.\n"
            "- Prefer quoting relevant facts verbatim and showing the reasoning chain.\n"
            "- Avoid bullet points unless they improve clarity for the trace."
        ),
    },
}

# UI/display -> canonical key mapping (keeps notebook stable even if UI labels change)
PERSONA_KEYMAP = {
    "Advogado": "advogado",
    "Gestor de Contrato": "gestor",
    "Executivo de negócio": "executivo",
    "Técnico": "tecnico",
    "advogado": "advogado",
    "gestor": "gestor",
    "executivo": "executivo",
    "tecnico": "tecnico",
}

def get_persona_key(persona_name: str) -> str:
    if persona_name is None:
        return "gestor"
    return PERSONA_KEYMAP.get(persona_name, persona_name.strip().lower())

def get_persona_cfg(persona_name: str) -> dict:
    key = get_persona_key(persona_name)
    return PERSONA_SOT.get(key, PERSONA_SOT["gestor"])

# Backward-compatible derived dicts (do NOT edit directly)
PERSONA_GUIDANCE_EN = {cfg["label_pt"]: cfg["prompt_guidance_en"] for cfg in PERSONA_SOT.values()}
PERSONA_RESPONSE_GUIDELINES = {k: v["response"] for k, v in PERSONA_SOT.items()}

In [ ]:
import re

def _strip_bullets(text: str) -> str:
    lines = text.splitlines()
    cleaned = []
    for ln in lines:
        # remove common bullet markers
        cleaned.append(re.sub(r"^\s*[-*•]\s+", "", ln))
    return "\n".join(cleaned).strip()

def _remove_clause_lines(text: str) -> str:
    """Conservative heuristic to remove clause-heavy lines for Executive answers."""
    out = []
    for ln in text.splitlines():
        lnl = ln.lower()
        if ("cláusula" in lnl) or ("clause" in lnl) or re.search(r"\b\d+\.\d+\b", ln):
            continue
        out.append(ln)
    return "\n".join(out).strip()

def adapt_response_to_persona(
    raw_text: str,
    persona_name: str,
    metadata: dict | None = None,
    user_requested_clause_evidence: bool = False
) -> str:
    """Persona-aware post-processing. Does NOT change facts; only presentation."""
    cfg = get_persona_cfg(persona_name)
    rules = cfg["response"]
    text = (raw_text or "").strip()

    # Avoid bullets by default (except when trace benefits)
    if rules.get("avoid_bullets", True):
        text = _strip_bullets(text)

    key = get_persona_key(persona_name)

    # EXECUTIVO: prosa curta; sem cláusulas por padrão
    if key == "executivo":
        if not user_requested_clause_evidence and not rules.get("show_clauses", False):
            text = _remove_clause_lines(text)

        # Keep first paragraph as digest
        parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        if parts:
            text = parts[0]

        return text[:900].rstrip()

    # GESTOR: recomendações úteis (sem "delegar a decisão")
    if key == "gestor" and rules.get("recommendations", False):
        low = text.lower()
        has_actions = re.search(r"recomenda|sugere|próximos passos|ações|oriente|proceda|deve", low) is not None

        if not has_actions:
            # Se a resposta é UNKNOWN/indeterminada, orientar coleta de evidências (sem dizer "verifique para determinar")
            if re.search(r"unknown|indetermin|não\s+pode\s+ser\s+determinado|não\s+foi\s+possível\s+determinar", low):
                text += (
                    "\n\nPara o gestor: ainda faltam evidências para classificar a criticidade. "
                    "Recomenda-se registrar explicitamente (no processo e/ou no instrumento) se o fornecedor terá "
                    "(i) acesso a informações sensíveis e/ou (ii) acesso administrativo/privilegiado a TI, conforme os critérios do semantic_ground."
                )
            else:
                text += (
                    "\n\nPara o gestor: recomenda-se registrar a classificação e sua justificativa, validar controles de acesso "
                    "(dados e privilégios), e ajustar o acompanhamento/monitoramento do contrato ao nível de criticidade identificado."
                )

        if rules.get("avoid_bullets", True):
            text = _strip_bullets(text)
        return text

    # TÉCNICO: acrescentar trace quando disponível
    if key == "tecnico" and rules.get("explain_reasoning", False) and metadata:
        engine = metadata.get("engine") or metadata.get("reasoning_engine") or "N/A"
        route = metadata.get("route") or metadata.get("routing") or "N/A"
        limits = metadata.get("limitations") or "N/A"
        trace = (
            "\n\n[Explicação Técnica]\n"
            f"- engine: {engine}\n"
            f"- routing: {route}\n"
            f"- limitations: {limits}"
        )
        return text + trace

    return text

In [ ]:
# NOTE: PERSONA_GUIDANCE_EN is now derived from PERSONA_SOT (single source of truth),
# defined earlier in the notebook. Do not redefine it here.

# Construção do prompt
from typing import List, Optional
model_name = "gemini-2.0-flash"
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'))

BASE_RULES_EN = """You are an assistant for contract analysis with formal grounding in a structured knowledge base.

Primary frame of reference:
- The semantic_ground is the primary frame of reference and MUST be applied as written.
- Do NOT introduce new requirements, new rules, or new labels that are not present in the semantic_ground.

General knowledge (allowed, but controlled):
- You MAY use general domain knowledge as an AUXILIARY bridge to map contract/UFO facts to semantic_ground requirements.
- If you use general knowledge, explicitly label it as INFERRED (not an explicit fact) and explain why it applies.
- Provide a confidence level for INFERRED bridges (high/medium/low).
- General knowledge must NEVER override explicit evidence in the provided artifacts.

Anti-escape rule:
- You must provide a concrete derived result whenever possible.
- Do NOT answer with generic statements like “it is determined based on requirements” without stating:
  (a) which requirements are satisfied / not satisfied / unknown,
  (b) the derived classification (or UNKNOWN),
  (c) a short justification grounded in the artifacts (and labeled INFERRED when applicable).
- Only answer “cannot be determined” / UNKNOWN if there is insufficient information even for a reasoned inference.
  In that case, explicitly list what is missing.

Meta-statements inside files:
- If a contract file contains analysis-like statements (e.g., “cannot be determined”, “information not found”),
  treat them as non-authoritative notes; prefer raw contract/UFO facts and semantic_ground rules.

CRITICALITY PROTOCOL (apply ONLY if the question is about criticality / 'criticidade'):
1) Identify the applicable derive_criticidade rule(s) from semantic_ground.
2) For each prerequisite requirement in those rules, decide: satisfied / not_satisfied / unknown.
3) For each decision, justify briefly using:
   - EVIDENCE: explicit contract/UFO facts or clauses, OR
   - INFERRED (confidence): a general-knowledge bridge that maps facts to the semantic_ground requirement.
4) Apply the derive_criticidade rule(s) and output: criticidade = <level> (or UNKNOWN).
5) If no rule can be applied, output UNKNOWN and list the unknown requirements.
Do NOT skip steps 2–4.
""".strip()

def persona_block(persona: str) -> str:
    p = (persona or "Gestor de Contrato").strip()
    return PERSONA_GUIDANCE_EN.get(p, PERSONA_GUIDANCE_EN["Gestor de Contrato"])

def build_direct_prompt(question: str, attachment: AttachmentText, persona: str, last_context: Optional[str] = None) -> str:
    # last_context can be used to disambiguate deictic questions (e.g., 'this contract').
    ctx_line = f"Context: The user refers to the previously selected contract file '{last_context}'.\n" if last_context else ""
    return f"""{BASE_RULES_EN}

{persona_block(persona)}

{ctx_line}SEMANTIC REQUIREMENTS (formal facts; definitions and interpretation criteria):
{SEMANTIC_GROUND_PL}

CONTRACT FILE (formal facts): {attachment.filename}
---
{attachment.text}
---

USER QUESTION:
{question}

Answer in Portuguese. Follow the selected persona strictly. If evidence is missing, say you did not find evidence in the provided facts.
Persona-specific formatting:
- Prefer continuous prose; avoid bullet points unless necessary.
- For non-Technical personas, do NOT show predicate syntax.
- Evidence style MUST follow the selected persona rules (Executive: no clause IDs/quotes unless explicitly requested).
- For Técnico, you may quote facts verbatim and mention predicates/routing.
"""

def answer_direct(question: str, attachment: AttachmentText, persona: str, last_context: Optional[str] = None) -> str:
    prompt = build_direct_prompt(question, attachment, persona, last_context=last_context)
    resp = client.models.generate_content(model=model_name, contents=prompt)
    return getattr(resp, "text", "").strip()

def build_semreq_prompt(question: str, persona: str) -> str:
    return f"""{BASE_RULES_EN}

{persona_block(persona)}

You have access to:
1) The CONTRACT MODEL (formal facts extracted from the contract), when provided.
2) The SEMANTIC GROUND (definitions, classifications, decision criteria and rules).

Instructions:
- Always use the CONTRACT MODEL to identify factual evidence about the specific contract.
- Use the SEMANTIC GROUND to interpret, classify, and justify those facts.
- Do NOT invent contract facts that are not present in the contract model.
- If the contract model does not contain enough information to answer conclusively, explicitly state this limitation.
- Only ignore the contract model if the user explicitly asks a generic or semantic-only question (e.g., 'sr' commands).

SEMANTIC REQUIREMENTS (formal facts):
---
{SEMANTIC_GROUND_PL}
---

USER QUESTION:
{question}

Answer in Portuguese. Follow the selected persona strictly. If evidence is missing, say you did not find evidence in the provided facts.
Persona-specific formatting:
- Prefer continuous prose; avoid bullet points unless necessary.
- For non-Technical personas, do NOT show predicate syntax.
- Evidence style MUST follow the selected persona rules (Executive: no clause IDs/quotes unless explicitly requested).
- For Técnico, you may quote facts verbatim and mention predicates/routing.
Prefer listing items as bullets when it is a "list/which" type question.
"""

def answer_semreq(question: str, persona: str) -> str:
    prompt = build_semreq_prompt(question, persona)
    resp = client.models.generate_content(model=model_name, contents=prompt)
    return getattr(resp, "text", "").strip()

# ---- Prompts para estratégia Map-Reduce (reutiliza o mesmo contrato epistemológico) ----

def build_map_prompt(question: str, attachment: AttachmentText, persona: str) -> str:
    """Prompt do MAP: decide relevância e extrai resposta/evidências por contrato (um arquivo por vez).
    Importante: NÃO cria um novo conjunto de regras; reutiliza BASE_RULES_EN e personas.
    """
    return f"""{BASE_RULES_EN}

{persona_block(persona)}

You are analyzing ONE contract represented as structured facts (single file).

Goal of the MAP step:
1) Decide if THIS contract is relevant to the question.
2) If relevant, extract a short answer and evidence ONLY from the facts below.

SEMANTIC REQUIREMENTS (formal facts; definitions and interpretation criteria):
{SEMANTIC_GROUND_PL}

TASK:
Return STRICT JSON in Portuguese with the following schema:
{{
  "relevant": true/false,
  "answer": "texto curto",
  "evidence": ["evid1", "evid2", "..."]
}}

RULES:
- Do NOT use any prior knowledge. Use ONLY the facts provided below.
- If not relevant, set relevant=false, answer="" and evidence=[].
- Evidence items must respect persona formatting:
  - For non-Technical personas: DO NOT use predicate syntax; reference clause identifiers and quote/paraphrase clause text/notes.
  - For Técnico: you may quote Prolog facts verbatim and mention predicates.

QUESTION:
{question}

CONTRACT FILE (formal facts): {attachment.filename}
---
{attachment.text}
---
JSON:
"""

def build_reduce_prompt(question: str, hits: List['MapHit'], persona: str) -> str:
    """Prompt do REDUCE: agrega resultados do MAP sem inventar nada."""
    # Serializa hits para o modelo (texto simples, sem perder evidências)
    serialized = []
    for h in hits:
        if not h.relevant:
            continue
        serialized.append({
            "filename": h.filename,
            "answer": h.answer,
            "evidence": h.evidence[:12],
        })
    return f"""{BASE_RULES_EN}

{persona_block(persona)}

You are aggregating MAP results from multiple contract files.
CRITICAL: You MUST NOT invent evidence or new facts. Use ONLY the MAP outputs below.

USER QUESTION:
{question}

MAP OUTPUTS (list of relevant files with extracted answers/evidence):
{json.dumps(serialized, ensure_ascii=False, indent=2)}

TASK:
Produce a single final answer in Portuguese following the selected persona.
- Mention which files support the conclusion (filenames).
- Use only the evidence lines provided in MAP OUTPUTS.
- If there are conflicts across files, explain the conflict and what evidence supports each side.

Formatting:
- Prefer bullet points when listing multiple contracts/files.
- For non-Technical personas, keep it natural (no predicate syntax).
- For Técnico, you may keep the evidence more literal.
"""

def answer_map(question: str, attachment: AttachmentText, persona: str) -> dict:
    prompt = build_map_prompt(question, attachment, persona)
    resp = client.models.generate_content(model=model_name, contents=prompt)
    return _safe_json_loads(getattr(resp, "text", "") or "") or {}

def answer_reduce(question: str, hits: List['MapHit'], persona: str) -> str:
    prompt = build_reduce_prompt(question, hits, persona)
    resp = client.models.generate_content(model=model_name, contents=prompt)
    return getattr(resp, "text", "").strip()


## 5) Estratégia Map-Reduce (perguntas globais)

Motivação: quando a pergunta exige olhar vários contratos, o volume total de fatos pode exceder limites de janela/saída.

- **MAP**: roda contrato a contrato, decide relevância e extrai resposta + evidência.
- **REDUCE**: agrega os MAP outputs sem inventar fatos, mantendo a persona.

In [ ]:
# Implementação da estratégia Map-Reduce
# Motivação: contornar limites de janela/volume quando a pergunta exige olhar múltiplos contratos.

@dataclass
class MapHit:
    filename: str
    relevant: bool
    answer: str
    evidence: List[str]

MAP_CACHE: Dict[Tuple[str, str, str], MapHit] = {}

def map_over_contract(
    question: str,
    attachment: AttachmentText,
    persona: str,
    sleep_seconds: float = 0.0,
) -> MapHit:
    """Executa o passo MAP para UM contrato (um arquivo Prolog)."""
    cache_key = (question, attachment.filename, persona)
    if cache_key in MAP_CACHE:
        return MAP_CACHE[cache_key]

    parsed = answer_map(question, attachment, persona)
    if not parsed:
        hit = MapHit(
            filename=attachment.filename,
            relevant=False,
            answer="Falha ao interpretar a resposta do modelo no MAP.",
            evidence=[],
        )
    else:
        hit = MapHit(
            filename=attachment.filename,
            relevant=bool(parsed.get("relevant", False)),
            answer=str(parsed.get("answer", "") or ""),
            evidence=list(parsed.get("evidence", []) or []),
        )

    MAP_CACHE[cache_key] = hit
    if sleep_seconds and sleep_seconds > 0:
        time.sleep(sleep_seconds)
    return hit

def reduce_hits(question: str, hits: List[MapHit], persona: str) -> str:
    """Agrega resultados do MAP, preferindo um REDUCE orientado a persona (LLM) sem inventar fatos."""
    rel = [h for h in hits if h.relevant]
    if not rel:
        return "Não encontrei contratos relevantes nos arquivos analisados."
    # Usa LLM para agregar mantendo o mesmo contrato epistemológico (BASE_RULES + persona)
    return answer_reduce(question, rel, persona)

def map_reduce_answer(question: str, attachments: List[AttachmentText], persona: str, sleep_seconds: float = 0.0) -> str:
    hits: List[MapHit] = []
    for a in attachments:
        hits.append(map_over_contract(question, a, persona, sleep_seconds=sleep_seconds))
    return reduce_hits(question, hits, persona)

## 6) Interface (Gradio)

A interface permite:
- selecionar a persona,
- escolher um contrato (quando aplicável),
- fazer perguntas diretas, semreq ou globais (via router).

In [ ]:
# Interface Gradio: roteia pergunta e chama DIRECT / SEMREQ / MAP-REDUCE.
# Apresenta interface (Gradio)
import gradio as gr
import re

LAST_SELECTED_FILENAME = None

def _user_requested_clause_evidence(message: str) -> bool:
    t = (message or "").lower()
    # Portuguese + English cues
    cues = [
        "cláusula", "clausula", "cláusulas", "clausulas",
        "trecho", "citar", "cite", "evidência", "evidencias", "evidence",
        "texto da cláusula", "id da cláusula", "identificador da cláusula",
        "mostrar as cláusulas", "mostre as cláusulas", "mostre os trechos",
        "quote", "quotation"
    ]
    return any(c in t for c in cues)

# Sleep para evitar rate limit
MAP_SLEEP_SECONDS = 0.25

# -----------------------------
# Meta-commands (UI / introspection)
# -----------------------------
_RE_MAN = re.compile(r"^\s*man\s*$", re.IGNORECASE)
_RE_LS  = re.compile(r"^\s*ls\s*$", re.IGNORECASE)
_RE_CAT = re.compile(r"^\s*cat\s+([A-Za-z0-9_\-().]+\.pl)\s*$", re.IGNORECASE)
_RE_SR  = re.compile(r"^\s*sr(\s+.+)?$", re.IGNORECASE)

def _render_contracts_index(attachments):
    names = [a.filename for a in attachments]
    if not names:
        return "Nenhum contrato (.pl) foi carregado."
    lines = ["Contratos carregados (arquivos .pl):"]
    lines += [f"- {n}" for n in names]
    return "\n".join(lines)

def _find_attachment_by_name(attachments, filename: str):
    fn = (filename or "").strip()
    if not fn:
        return None
    # match case-insensitively, but return the original
    for a in attachments:
        if a.filename.lower() == fn.lower():
            return a
    return None

def handle_meta_command(message: str, attachments, semantic_requirements_path: str):
    t = (message or "").strip()

    # man
    if _RE_MAN.match(t):
        return (
            "Comandos disponíveis:\n\n"
            "man                     - mostra esta ajuda\n"
            "ls                      - lista contratos (.pl) carregados\n"
            "cat <arquivo>.pl        - mostra o conteúdo de um contrato\n"
            "sr                      - mostra os requisitos semânticos (SEMREQ)\n"
            "sr <pergunta>           - responde usando SOMENTE o SEMREQ\n\n"
            "Escopo:\n"
            "- Sem '/all': consulta apenas o contrato atual\n"
            "- Com '/all': consulta todos os contratos (map-reduce)\n"
        )

    # ls
    if _RE_LS.match(t):
        if not attachments:
            return "Não há contratos (.pl) carregados."
        return "Contratos disponíveis:\n- " + "\n- ".join(a.filename for a in attachments)

    # cat <arquivo>.pl
    m_cat = _RE_CAT.match(t)
    if m_cat:
        fname = m_cat.group(1)
        for a in attachments:
            if a.filename.lower() == fname.lower():
                return f"```prolog\n{a.text}\n```"
        return f"Arquivo '{fname}' não encontrado."

    # sr ou sr <pergunta>
    m_sr = _RE_SR.match(t)
    if m_sr:
        # sr puro → dump SEMREQ
        if not m_sr.group(1):
            if not os.path.exists(semantic_requirements_path):
                return "Arquivo semantic_requirements.pl não encontrado."
            with open(semantic_requirements_path, "r", encoding="utf-8") as f:
                return f"```prolog\n{f.read()}\n```"

        # sr <pergunta> → deixa o chat_fn tratar
        return "__SR_QUERY__"

    return None


def chat_fn(message, history, persona):
    global LAST_SELECTED_FILENAME
    if history is None:
        history = []

    # ---- meta commands (must run BEFORE routing) ----
    cmd_resp = handle_meta_command(message, ATTACHMENTS, SEMANTIC_GROUND_PATH)

    if cmd_resp == "__SR_QUERY__":
        # remove o prefixo 'sr'
        question = message.strip()[2:].strip()
        return adapt_response_to_persona(answer_semreq(question, persona), persona_name=persona, metadata={'engine':'gemini','route':'SEMREQ'}, user_requested_clause_evidence=_user_requested_clause_evidence(message))

    if cmd_resp is not None:
        return cmd_resp

    # Select attachments based on enhanced router (uses history + last selection)
    selected, new_last, route = select_attachments_for_question(
        message,
        ATTACHMENTS,
        history,
        LAST_SELECTED_FILENAME,
    )

    try:
        if route == "semreq":
            # Answer only from semantic requirements
            return adapt_response_to_persona(answer_semreq(message, persona), persona_name=persona, metadata={'engine':'gemini','route':'SEMREQ'}, user_requested_clause_evidence=_user_requested_clause_evidence(message))

        if route == "direct":
            if not selected:
                ocs_id = extract_ocs_from_text(message)
                if ocs_id:
                    return f"Não encontrei o arquivo Prolog do contrato {ocs_id} nos anexos."
                return (
                    "Não consegui identificar qual contrato você está se referindo. "
                    "Informe o OCS (ex.: 'OCS 048/2022 ...') ou selecione um contrato. "
                    "Se você quiser consultar TODOS os contratos, adicione '/all' à pergunta."
                )

            # contrato definido -> DIRECT normal
            LAST_SELECTED_FILENAME = new_last
            return adapt_response_to_persona(answer_direct(message, selected[0], persona, last_context=LAST_SELECTED_FILENAME), persona_name=persona, metadata={'engine':'gemini','route':'DIRECT'}, user_requested_clause_evidence=_user_requested_clause_evidence(message))

        # GLOBAL route -> Map-Reduce
        return adapt_response_to_persona(map_reduce_answer(message, selected, persona, sleep_seconds=MAP_SLEEP_SECONDS), persona_name=persona, metadata={'engine':'gemini','route':'MAP-REDUCE'}, user_requested_clause_evidence=_user_requested_clause_evidence(message))

    except Exception as e:
        return f"Erro ao processar a pergunta: {e}"


# -----------------------------
# UI: Chat + Instructions
# -----------------------------

INSTRUCTIONS_MD = """
## Instruções (comandos básicos)

Você pode digitar perguntas normais (em português) sobre os contratos anexados.

### Comandos
- `man`
  Mostra esta ajuda.

- `ls`
  Lista os contratos (.pl) carregados.

- `cat <arquivo>.pl`
  Mostra o conteúdo de um contrato específico.

- `sr`
  Mostra o conteúdo do arquivo de requisitos/ground semântico (SEMREQ).

- `sr <pergunta>`
  Responde usando **somente** o SEMREQ (sem olhar contratos).

### Escopo de consulta
- Pergunta normal (sem `/all`): tenta identificar **um** contrato relevante e responde usando ele.
- Pergunta com `/all`: consulta **todos** os contratos carregados (map-reduce).

### Exemplos
- `ls`
- `cat KOA_UFO_contract_ocs_018_2023.pl`
- `sr`
- `sr Quais são as perguntas de segurança da informação?`
- `O contrato prevê direito de auditoria?`
- `O contrato prevê direito de auditoria? /all`
"""

def _append_history(history, user_msg, bot_msg):
    history = history or []
    history.append((user_msg, bot_msg))
    return history

def ui_respond(message, history, persona):
    message = (message or "").strip()
    if not message:
        return history, ""
    bot_msg = chat_fn(message, history, persona)
    history = _append_history(history, message, bot_msg)
    return history, ""

def ui_clear():
    return []

with gr.Blocks(title="Contrato360 (Powered by KOA)") as demo:
    gr.Markdown("<h1 style='text-align: center;'>Contrato360 (Powered by KOA)</h1>")

    with gr.Row():
        btn_chat = gr.Button("Chat", variant="primary")
        btn_instructions = gr.Button("Instruções")

    with gr.Column(visible=True) as chat_panel:

        persona = gr.Dropdown(
            label="Persona",
            choices=["Advogado", "Gestor de Contrato", "Executivo de negócio", "Técnico"],
            value="Gestor de Contrato",
            interactive=True,
        )
        chatbot = gr.Chatbot(height=520)
        with gr.Row():
            msg = gr.Textbox(placeholder="Digite sua pergunta aqui… (use /all para consultar todos)", scale=4)
            send = gr.Button("Enviar", scale=1)
        with gr.Row():
            clear = gr.Button("Limpar conversa")

    with gr.Column(visible=False) as instructions_panel:
        gr.Markdown(INSTRUCTIONS_MD)
        back = gr.Button("Voltar para o Chat", variant="primary")

    # Wiring
    send.click(ui_respond, inputs=[msg, chatbot, persona], outputs=[chatbot, msg])
    msg.submit(ui_respond, inputs=[msg, chatbot, persona], outputs=[chatbot, msg])
    clear.click(ui_clear, inputs=None, outputs=[chatbot])

    def _show_instructions():
        return gr.update(visible=False), gr.update(visible=True)

    def _show_chat():
        return gr.update(visible=True), gr.update(visible=False)

    btn_instructions.click(_show_instructions, inputs=None, outputs=[chat_panel, instructions_panel])
    back.click(_show_chat, inputs=None, outputs=[chat_panel, instructions_panel])
    btn_chat.click(_show_chat, inputs=None, outputs=[chat_panel, instructions_panel])

demo.launch(debug=True)
